In [1]:
!pip install -q groq
print("groq installed ")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 2.2 MB/s eta 0:00:00
groq installed 


In [5]:
from google.colab import drive, userdata
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
import os
import sys

import re
import difflib
from groq import Groq



In [7]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("Groq key loaded")

Groq key loaded


**Write groq_fixer.py to Drive**

In [9]:
%%writefile /content/drive/MyDrive/code-debugger/backend/groq_fixer.py
import os, re
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY", ""))

SYSTEM_PROMPT = """You are an expert Python debugger. Respond in EXACTLY this format:

## Explanation
2-3 sentences describing the bug, why it happens, and which line causes it.

## Fixed Code
```python
<complete corrected code here>
```

Rules:
- Always return the FULL corrected code not just the changed lines
- If multiple bugs exist list all of them in Explanation
- Never add new features beyond fixing the bugs
- Always mention the line number where the bug occurs"""


def fix_with_groq(code: str, error_label: str) -> dict:
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": (
                        f"Bug type already classified as: {error_label}\n\n"
                        f"Debug this Python code:\n```python\n{code}\n```"
                    ),
                },
            ],
            temperature=0.1,
            max_tokens=1024,
        )
        raw = response.choices[0].message.content
        return {
            "explanation": _extract_section(raw, "## Explanation"),
            "fixed_code": _extract_code_block(raw),
            "raw": raw,
        }
    except Exception as e:
        return {
            "explanation": f"Groq error: {str(e)}",
            "fixed_code": code,
            "raw": "",
        }


def _extract_section(text: str, header: str) -> str:
    lines = text.split("\n")
    capturing, result = False, []
    for line in lines:
        if line.strip() == header:
            capturing = True
            continue
        if capturing and line.startswith("## "):
            break
        if capturing:
            result.append(line)
    return "\n".join(result).strip()


def _extract_code_block(text: str) -> str:
    match = re.search(r"```python\n(.*?)```", text, re.DOTALL)
    return match.group(1).strip() if match else ""

Writing /content/drive/MyDrive/code-debugger/backend/groq_fixer.py


**TEST the fixer**

In [13]:

sys.path.insert(0,"/content/drive/MyDrive/code-debugger/backend")
from groq_fixer import fix_with_groq

result=fix_with_groq(
    code="def greet(name)\n    print('Hello, ' + name)",
    error_label="Syntax Error"
)
print("EXPLANATION:")
print(result["explanation"])
print("\nFIXED CODE:")
print(result["fixed_code"])

EXPLANATION:
The bug in this code is a syntax error due to a missing colon at the end of the function definition. This error occurs on line 1, where the function `greet` is defined without a colon to indicate the start of the function body. The corrected code will include a colon at the end of the function definition.

FIXED CODE:
def greet(name):
    print('Hello, ' + name)


**Adding generation**

In [18]:
%%writefile -a /content/drive/MyDrive/code-debugger/backend/groq_fixer.py

import difflib

def generate_diff(original: str, fixed: str) -> str:
    if not fixed:
        return ""
    diff = difflib.unified_diff(
        original.splitlines(keepends=True),
        fixed.splitlines(keepends=True),
        fromfile="buggy.py",
        tofile="fixed.py",
        lineterm="",
    )
    return "\n".join(list(diff))

Appending to /content/drive/MyDrive/code-debugger/backend/groq_fixer.py


In [19]:
from groq_fixer import fix_with_groq, generate_diff

code = "def greet(name)\n    print('Hello, ' + name)"
result = fix_with_groq(code, "Syntax Error")
diff = generate_diff(code, result["fixed_code"])

print("DIFF:")
print(diff)

ImportError: cannot import name 'generate_diff' from 'groq_fixer' (/content/drive/MyDrive/code-debugger/backend/groq_fixer.py)

In [20]:
with open("/content/drive/MyDrive/code-debugger/backend/groq_fixer.py", "r") as f:
    print(f.read())

import os, re
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY", ""))

SYSTEM_PROMPT = """You are an expert Python debugger. Respond in EXACTLY this format:

## Explanation
2-3 sentences describing the bug, why it happens, and which line causes it.

## Fixed Code
```python
<complete corrected code here>
```

Rules:
- Always return the FULL corrected code not just the changed lines
- If multiple bugs exist list all of them in Explanation
- Never add new features beyond fixing the bugs
- Always mention the line number where the bug occurs"""


def fix_with_groq(code: str, error_label: str) -> dict:
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": (
                        f"Bug type already classified as: {error_label}\n\n"
                 

In [21]:
%%writefile /content/drive/MyDrive/code-debugger/backend/groq_fixer.py
import os, re, difflib
from groq import Groq

client = Groq(api_key=os.environ.get("GROQ_API_KEY", ""))

SYSTEM_PROMPT = """You are an expert Python debugger. Respond in EXACTLY this format:

## Explanation
2-3 sentences describing the bug, why it happens, and which line causes it.

## Fixed Code
```python
<complete corrected code here>
```

Rules:
- Always return the FULL corrected code not just the changed lines
- If multiple bugs exist list all of them in Explanation
- Never add new features beyond fixing the bugs
- Always mention the line number where the bug occurs"""


def fix_with_groq(code: str, error_label: str) -> dict:
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": (
                        f"Bug type already classified as: {error_label}\n\n"
                        f"Debug this Python code:\n```python\n{code}\n```"
                    ),
                },
            ],
            temperature=0.1,
            max_tokens=1024,
        )
        raw = response.choices[0].message.content
        return {
            "explanation": _extract_section(raw, "## Explanation"),
            "fixed_code": _extract_code_block(raw),
            "raw": raw,
        }
    except Exception as e:
        return {
            "explanation": f"Groq error: {str(e)}",
            "fixed_code": code,
            "raw": "",
        }


def generate_diff(original: str, fixed: str) -> str:
    if not fixed:
        return ""
    diff = difflib.unified_diff(
        original.splitlines(keepends=True),
        fixed.splitlines(keepends=True),
        fromfile="buggy.py",
        tofile="fixed.py",
        lineterm="",
    )
    return "\n".join(list(diff))


def _extract_section(text: str, header: str) -> str:
    lines = text.split("\n")
    capturing, result = False, []
    for line in lines:
        if line.strip() == header:
            capturing = True
            continue
        if capturing and line.startswith("## "):
            break
        if capturing:
            result.append(line)
    return "\n".join(result).strip()


def _extract_code_block(text: str) -> str:
    match = re.search(r"```python\n(.*?)```", text, re.DOTALL)
    return match.group(1).strip() if match else ""

Overwriting /content/drive/MyDrive/code-debugger/backend/groq_fixer.py


In [22]:
import importlib, sys

# Remove cached version
if "groq_fixer" in sys.modules:
    del sys.modules["groq_fixer"]

from groq_fixer import fix_with_groq, generate_diff

code = "def greet(name)\n    print('Hello, ' + name)"
result = fix_with_groq(code, "Syntax Error")
diff = generate_diff(code, result["fixed_code"])

print("DIFF:")
print(diff)

DIFF:
--- buggy.py
+++ fixed.py
@@ -1,2 +1,2 @@
-def greet(name)

+def greet(name):

     print('Hello, ' + name)
